# Solutions — Incremental Processing: New Files, Checkpoints, Watermarks, Upserts, Late Data

Exercise notebook: `08_data_engineering_python/05_incremental_processing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Which files are new?

**Working solution**

In [ ]:
def files_to_process(available, processed):
    return sorted(set(available) - set(processed))

**Explanation**

Set difference is the whole algorithm. ISO dates in partition names make lexicographic order equal chronological order — one reason partitions use `YYYY-MM-DD`.

**Complexity:** O(n log n).

**Common mistakes**

- Processing in arbitrary order (`set` iteration) → watermarks jump around.

## Exercise 2 — 🟡 Medium · A durable JSON checkpoint

**Working solution**

In [ ]:
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

class Checkpoint:
    def __init__(self, path):
        self.path = Path(path)
        self.processed, self.watermark = {}, None
        if self.path.exists():
            try:
                state = json.loads(self.path.read_text(encoding="utf-8"))
            except json.JSONDecodeError as exc:
                raise ValueError(f"corrupt checkpoint {self.path}") from exc
            self.processed = dict(state.get("processed", {}))
            self.watermark = state.get("watermark")

    def is_processed(self, key, checksum=None):
        if key not in self.processed:
            return False
        return checksum is None or self.processed[key] == checksum

    def mark(self, key, checksum):
        self.processed[key] = checksum

    def advance(self, watermark):
        if watermark is not None and (self.watermark is None or watermark > self.watermark):
            self.watermark = watermark

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        state = {"processed": self.processed, "watermark": self.watermark,
                 "updated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")}
        fd, tmp = tempfile.mkstemp(dir=self.path.parent, suffix=".tmp")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as f:
                json.dump(state, f, indent=2, sort_keys=True)
            os.replace(tmp, self.path)
        except BaseException:
            Path(tmp).unlink(missing_ok=True)
            raise

**Explanation**

The checkpoint is the pipeline's memory. Atomic save means a crash leaves either the old or the new state, never half a file. Failing loudly on corruption is a deliberate
choice: a silent reset would reprocess (and double-count) history.

**Complexity:** O(state size) per save.

**Common mistakes**

- Saving the checkpoint *before* the output is durably written → data loss on crash.
- Watermark moving backwards on a late file.

## Exercise 3 — 🟡 Medium · Watermark filtering

**Working solution**

In [ ]:
from datetime import datetime, timedelta

FMT = "%Y-%m-%dT%H:%M:%SZ"

def select_new(records, watermark, ts_field, lookback_seconds=0):
    if watermark is None:
        selected = list(records)
    else:
        cutoff = (datetime.strptime(watermark, FMT) - timedelta(seconds=lookback_seconds)).strftime(FMT)
        selected = [r for r in records if r[ts_field] > cutoff]
    candidates = [r[ts_field] for r in records] + ([watermark] if watermark else [])
    return selected, max(candidates) if candidates else None

**Explanation**

Fixed-width ISO UTC strings compare correctly as strings; parse only when doing arithmetic (the lookback). The watermark is monotonic by construction (`max` including the old one).

**Complexity:** O(n).

**Common mistakes**

- Using `>=` and reprocessing the boundary record every run without idempotent writes.
- Watermark from selected records only.

## Exercise 4 — 🟠 Hard · Upserts with stale-update protection

**Working solution**

In [ ]:
def upsert(target, changes, key, version_field):
    counts = {"inserted": 0, "updated": 0, "deleted": 0, "skipped_stale": 0}
    tombstones = {}
    for change in changes:
        k, version = change[key], change[version_field]
        current = target.get(k)
        known_version = current[version_field] if current else tombstones.get(k)
        if known_version is not None and version <= known_version:
            counts["skipped_stale"] += 1
            continue
        if change.get("_op") == "delete":
            if current is not None:
                del target[k]
                counts["deleted"] += 1
            tombstones[k] = version
            continue
        row = {f: v for f, v in change.items() if f != "_op"}
        if current is None:
            counts["inserted"] += 1
        else:
            counts["updated"] += 1
        target[k] = row
    return counts

**Explanation**

Version comparison makes the merge **commutative**: whatever order CDC events arrive in, the newest wins. Remembering *tombstones* (deleted key + version) stops an older
update from resurrecting a deleted row — the same reason Kafka compaction and Delta Lake keep delete markers.

**Complexity:** O(changes).

**Common mistakes**

- Last-write-wins by arrival order.
- Forgetting deletes can arrive before older updates.

## Challenge — 🔴 Very Hard · Late data and a reprocessing window

**Working solution**

In [ ]:
import copy
from datetime import date, timedelta

def update_daily_aggregates(state, events, allowed_lateness_days=2):
    new_state = copy.deepcopy(state)
    days = new_state.setdefault("days", {})
    watermark = new_state.get("watermark")
    cutoff = (date.fromisoformat(watermark) - timedelta(days=allowed_lateness_days)).isoformat() if watermark else None
    changed, dropped = set(), 0
    seen = {d: set(v.get("event_ids", [])) for d, v in days.items()}
    max_date = watermark
    for e in events:
        if cutoff is not None and e["date"] < cutoff:
            dropped += 1
            continue
        bucket = days.setdefault(e["date"], {"revenue": 0.0, "events": 0, "event_ids": []})
        ids = seen.setdefault(e["date"], set())
        if e["event_id"] in ids:
            continue
        ids.add(e["event_id"])
        bucket["event_ids"].append(e["event_id"])
        bucket["revenue"] = round(bucket["revenue"] + e["amount"], 2)
        bucket["events"] += 1
        changed.add(e["date"])
        if max_date is None or e["date"] > max_date:
            max_date = e["date"]
    new_state["watermark"] = max_date
    return new_state, sorted(changed), dropped

**Explanation**

The watermark defines how late is 'too late'; the lateness window bounds state size and how far back outputs can change. Returning *changed dates* lets the caller rewrite
only affected output partitions — the core of efficient incremental recomputation. Note the cutoff is computed from the watermark **before** the batch.

**Complexity:** O(events).

**Common mistakes**

- Accepting arbitrarily late data → unbounded state and rewriting all of history.
- Double-counting replayed events.